In [26]:
import torch
import torch.nn.functional as F


DIM = 2048
NUM_PRUNED = 10

# 2. Create two random tensors of size 2048
v1 = torch.load('/workspace/CCE_NLI/hidden_state_student.pth')[0][8]
v2 = torch.load('/workspace/CCE_NLI/hidden_state_teacher.pth')[0][8]

# 3. Choose 10 random indices to zero out across both tensors
zero_indices = torch.randperm(DIM)[:NUM_PRUNED]

# Create a boolean mask tracking active features (True for active, False for zeroed)
active_mask = torch.ones(DIM, dtype=torch.bool)
active_mask[zero_indices] = False

# Apply the zeros to both tensors
v1_zeroed = v1.clone()
v2_zeroed = v2.clone()
v1_zeroed[zero_indices] = 0.0
v2_zeroed[zero_indices] = 0.0

# ==========================================
# CASE 1: Standard LayerNorm (Includes Zeros)
# ==========================================
v1_ln_include = F.layer_norm(v1_zeroed, (DIM,))
v2_ln_include = F.layer_norm(v2_zeroed, (DIM,))

# Calculate MSE for the included case
mse_include = F.mse_loss(v1_ln_include, v2_ln_include, reduction='mean')

# ==========================================
# CASE 2: Exclude Zeros LayerNorm (Masked)
# ==========================================
def layer_norm_exclude_zeros(x, mask, eps=1e-5):
    # Pull out only the elements where mask is True
    active_elements = x[mask]
    
    # Calculate mean and variance ONLY from active components
    mean = active_elements.mean()
    var = active_elements.var(unbiased=False) # Keep match with PyTorch LayerNorm
    
    # Normalize the entire tensor using the active stats
    normalized = (x - mean) / torch.sqrt(var + eps)
    
    # Force the pruned indices to stay hard zero
    normalized[~mask] = 0.0
    return normalized

v1_ln_exclude = layer_norm_exclude_zeros(v1_zeroed, active_mask)
v2_ln_exclude = layer_norm_exclude_zeros(v2_zeroed, active_mask)

# Calculate MSE for the excluded case
mse_exclude = F.mse_loss(v1_ln_exclude, v2_ln_exclude, reduction='mean')

# ==========================================
# PRINT RESULTS
# ==========================================
print(f"Total Dimensions:                 {DIM}")
print(f"Zeroed-out Indices Count:         {NUM_PRUNED}")
print("-" * 50)
print(f"MSE (Including Zeros in LN):      {mse_include.item():.8f}")
print(f"MSE (Excluding Zeros in LN):      {mse_exclude.item():.8f}")
print("-" * 50)
print(f"v1_ln (Included) at zero index:   {v1_ln_include[zero_indices[0]].item():.6f}")
print(f"v1_ln (Excluded) at zero index:   {v1_ln_exclude[zero_indices[0]].item():.6f}")


Total Dimensions:                 2048
Zeroed-out Indices Count:         10
--------------------------------------------------
MSE (Including Zeros in LN):      1.08667934
MSE (Excluding Zeros in LN):      1.08137310
--------------------------------------------------
v1_ln (Included) at zero index:   0.003151
v1_ln (Excluded) at zero index:   0.000000


In [8]:

# 1. Define your input tensor (e.g., Batch size = 1, Sequence length = 1, Hidden dim = 3)
input_tensor = torch.tensor([[[3.0, -4.0, 5.0,0.0,0.0]]])

# 2. Define the shape of the dimensions you want to normalize over
# This targets the last dimension (the 3 elements)
normalized_shape = (input_tensor.size(-1),) 

# Option A: Pure normalization without any learnable weights (returns standard normal)
output_pure = F.layer_norm(input_tensor, normalized_shape)

# Option B: Replicating a full LayerNorm layer with custom weight and bias parameters
weight = torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0])  # Scale multiplier (gamma)
bias = torch.tensor([0.0, 0.0, 0.0, 0.0, 0.0])    # Shift addition (beta)
eps = 1e-5

output_with_weights = F.layer_norm(input_tensor, normalized_shape, weight, bias, eps)

# Print results
print(f"Original Input:       {input_tensor.tolist()}")
print(f"Pure LayerNorm:       {output_pure.tolist()}")
print(f"LayerNorm w/ Weights: {output_with_weights.tolist()}")


Original Input:       [[[3.0, -4.0, 5.0, 0.0, 0.0]]]
Pure LayerNorm:       [[[0.7190922498703003, -1.5689284801483154, 1.3728123903274536, -0.26148808002471924, -0.26148808002471924]]]
LayerNorm w/ Weights: [[[0.7190922498703003, -1.5689284801483154, 1.3728123903274536, -0.26148808002471924, -0.26148808002471924]]]


In [21]:
v1 = torch.load('/workspace/CCE_NLI/hidden_state_student.pth')